In [7]:
import os
import shutil

# 0. Bulletproof Setup: Check for repo and enforce working directory
repo_path = "/kaggle/working/rl_sf"
if not os.path.exists(repo_path):
    print("--> Repository missing. Cloning now...")
    !git clone -b optimization https://github.com/flaviogeuforbio/rl-with-sf-for-mujoco {repo_path}

# Change directory explicitly to where the script lives
%cd {repo_path}

--> Repository missing. Cloning now...
Cloning into '/kaggle/working/rl_sf'...
remote: Enumerating objects: 3632, done.
remote: Total 3632 (delta 0), reused 0 (delta 0), pack-reused 3632 (from 3)
Receiving objects: 100% (3632/3632), 625.29 MiB | 41.08 MiB/s, done.
Resolving deltas: 100% (655/655), done.
Updating files: 100% (2397/2397), done.
/kaggle/working/rl_sf


In [8]:
import torch
print("torch:", torch.version)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

torch: <module 'torch.version' from '/usr/local/lib/python3.12/dist-packages/torch/version.py'>
cuda available: True
device: Tesla T4


In [9]:
%ls

ActorCritic.py                        quick_test.py
args_plot_average_results_walker.txt  render_agent.py
artifacts/                            requirements.txt
diagnose_feature_scales.py            run_behavioral_diagnostics_mod.py
diagnose_psi.py                       run_behavioral_diagnostics.py
diagnose_rollout_dynamics.py          run_psi_diagnostic_mod.py
OLD_2_train_cheetah_walker.py         run_psi_diagnostic.py
OLD_train_cheetah_walker.py           train_cheetah_walker.py
plot_average_results.py               train_sf_ddpg.py
PlotResults.py                        transfer_vs_scratch_comparison.pdf
plot_rollout_timeseries.py            transfer_vs_scratch_comparison.png
__pycache__/                          utils.py
QuickPlot.py                          zero_shot_eval.py


In [10]:
!python -m pip install mujoco

In [11]:
import os
import shutil

# --- PHASE 1 SMOKE TEST: INITIAL RUN ---

STEPS_PER_PHASE = "4000"  
SAVE_FREQ = "1000"
RUN_STEPS_LIMIT = "2000"
GAMMA_VAL = "0.99"
LAMBDA_Q = "1.0" 
LAMBDA_VEC = "1.0" 
#RESUME_DIR = "/kaggle/input/datasets/adrianoarceri/checkpoint" # change the last name accoording to how you name the dataset, and the previous folder is your username
# Here I don't need the resume_dir!!
# Removed stepsxphase to keep the directory name constant across resumed chunks
RUN_NAME_SEQ = f"Walker_from_scratch_gamma_{GAMMA_VAL.replace('.', '_')}_lq_{LAMBDA_Q.replace('.', '_')}_lvec_{LAMBDA_VEC.replace('.', '_')}_stepsxphase_{STEPS_PER_PHASE}"

kaggle_output_folder = "/kaggle/working/scratch_learning_long_run"
os.makedirs(kaggle_output_folder, exist_ok=True)

local_path_seq = f"/kaggle/working/rl_sf/artifacts/walker/{RUN_NAME_SEQ}"

seed=1 
print(f"\n--- EXECUTING SEED {seed} ---")

print("-> Running scratch Training...")
!python -u train_cheetah_walker.py --steps_per_phase {STEPS_PER_PHASE} --save_freq {SAVE_FREQ} --baseline --run_name {RUN_NAME_SEQ} --gamma {GAMMA_VAL} --lambda_q {LAMBDA_Q} --lambda_vec {LAMBDA_VEC} --seed {seed} --run_steps_limit {RUN_STEPS_LIMIT} --walker_only

if os.path.exists(local_path_seq):
    final_dest_seq = os.path.join(kaggle_output_folder, RUN_NAME_SEQ)
    shutil.copytree(local_path_seq, final_dest_seq, dirs_exist_ok=True)
    print(f"--> Scratch data saved in: {final_dest_seq}")

print("\n--> Zipping results for download...")
%cd /kaggle/working/
!zip -r scratch_learning_long_run.zip scratch_learning_long_run/
%cd /kaggle/working/rl_sf


--- EXECUTING SEED 1 ---
-> Running scratch Training...
Training SF-DDPG...
--- Starting Phase 1 (Walker) ---
Step: 196 | Episodes: 10 | Avg Return: 0.02 | C Loss: 0.0000 | Q Loss: 0.0000 | Vec Loss: 0.0000
Step: 383 | Episodes: 20 | Avg Return: -0.02 | C Loss: 0.0000 | Q Loss: 0.0000 | Vec Loss: 0.0000
Step: 554 | Episodes: 30 | Avg Return: 0.13 | C Loss: 0.0000 | Q Loss: 0.0000 | Vec Loss: 0.0000
Step: 737 | Episodes: 40 | Avg Return: 0.00 | C Loss: 0.0000 | Q Loss: 0.0000 | Vec Loss: 0.0000
Step: 950 | Episodes: 50 | Avg Return: 0.15 | C Loss: 0.0000 | Q Loss: 0.0000 | Vec Loss: 0.0000
--> [SAVE] Checkpoint saved at Phase 1, Step 1000
Step: 1294 | Episodes: 60 | Avg Return: 0.18 | C Loss: 0.0641 | Q Loss: 0.0026 | Vec Loss: 0.0615
--> [SAVE] Checkpoint saved at Phase 1, Step 2000
Chunk limit reached. Exiting safely without closing the phase.
Training DDPG...
--- Starting Task 2 (Walker Forward) from Scratch ---
Step: 196 | Episodes: 10 | Avg Return: 0.02 | C Loss: 0.0000
Step: 383 